# Update Dates
#### Added 10 Years

## Connection to **Spark**

In [ ]:
import os
import sys
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

# 1. Set PYSPARK_SUBMIT_ARGS to match your working batch file launcher
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

# 2. Ensure Python paths align for the worker processes
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = "C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

# staging_table_name = "staging.Integration.employee_Staging"
# wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)

spark
# spark.sql("SHOW CATALOGS").show(truncate=False)
# spark.sql("SHOW NAMESPACES IN reporting").show(truncate=False)
# spark.sql("SHOW DATABASES IN reporting").show(truncate=False)
# spark.sql("SHOW TABLES IN reporting.dimension").show(truncate=False)
for ns in spark.sql("SHOW NAMESPACES IN reporting").collect():
    namespace = ns["namespace"]
    # print(f"\nNamespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)

# Update Days

## Add Day to make Data Current

In [ ]:
spark.sql("""
UPDATE reporting.dimension.date
SET 
    Date = date_add(Date, 57),
    Day_Number = day(date_add(Date, 57)),
    Day = date_format(date_add(Date, 57), 'EEEE'),
    Month = date_format(date_add(Date, 57), 'MMMM'),
    Short_Month = date_format(date_add(Date, 57), 'MMM'),
    Calendar_Month_Number = month(date_add(Date, 57)),
    Calendar_Month_Label = date_format(date_add(Date, 57), 'yyyy-MMM'),
    Calendar_Year = year(date_add(Date, 57)),
    Calendar_Year_Label = concat('CY', year(date_add(Date, 57))),
    ISO_Week_Number = weekofyear(date_add(Date, 57))
""")

spark.sql("""
UPDATE reporting.fact.transaction
SET 
    Date_Key = date_add(Date_Key, 57)
""")

spark.sql("""
UPDATE reporting.fact.movement
SET 
    Date_Key = date_add(Date_Key, 57)
""")

spark.sql("""
UPDATE reporting.fact.order
SET 
    Order_Date_Key = date_add(Order_Date_Key, 57),
    Picked_Date_Key = date_add(Picked_Date_Key, 57)
""")

spark.sql("""
UPDATE reporting.fact.purchase
SET 
    Date_Key = date_add(Date_Key, 57)
""")

spark.sql("""
UPDATE reporting.fact.sale
SET 
    Invoice_Date_Key = date_add(Invoice_Date_Key, 57),
    Delivery_Date_Key = date_add(Delivery_Date_Key, 57)
""")

# Max Date  = 07/27/2026

## Updates Dates Add Months

### Date Table

In [ ]:
spark.sql("""
UPDATE reporting.dimension.date
SET 
    Date = add_months(Date, 120),
    Day_Number = day(add_months(Date, 120)),
    Day = date_format(add_months(Date, 120), 'EEEE'),
    Month = date_format(add_months(Date, 120), 'MMMM'),
    Short_Month = date_format(add_months(Date, 120), 'MMM'),
    Calendar_Month_Number = month(add_months(Date, 120)),
    Calendar_Month_Label = date_format(add_months(Date, 120), 'yyyy-MMM'),
    Calendar_Year = year(add_months(Date, 120)),
    Calendar_Year_Label = concat('CY', year(add_months(Date, 120))),
    ISO_Week_Number = weekofyear(add_months(Date, 120))
""")

### Fact Table transaction

In [ ]:
spark.sql("""
UPDATE reporting.fact.transaction
SET 
    Date_Key = add_months(Date_Key, 120)
""")

### Fact Table movement

In [ ]:
spark.sql("""
UPDATE reporting.fact.movement
SET 
    Date_Key = add_months(Date_Key, 120)
""")

### Fact Table Order

In [ ]:
spark.sql("""
UPDATE reporting.fact.order
SET 
    Order_Date_Key = add_months(Order_Date_Key, 120),
    Picked_Date_Key = add_months(Picked_Date_Key, 120)
""")

### Fact Table Purchase

In [ ]:
spark.sql("""
UPDATE reporting.fact.purchase
SET 
    Date_Key = add_months(Date_Key, 120)
""")

### Fact Table Sale

In [ ]:
spark.sql("""
UPDATE reporting.fact.sale
SET 
    Invoice_Date_Key = add_months(Invoice_Date_Key, 120),
    Delivery_Date_Key = add_months(Delivery_Date_Key, 120)
""")

In [ ]:
spark.stop()